# Transformer-Based Sentiment Analyzer

This notebook demonstrates sentiment analysis using state-of-the-art transformer models from Hugging Face.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install transformers torch datasets accelerate

In [ ]:
import pandas as pd
import numpy as np
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
# Load the same dataset used in the dictionary-based approach
df = pd.read_csv("DATA/raw/review_corpus.tsv", sep="\t")
df.head()

In [ ]:
ratings = list(df['rating'])
reviews = list(df['review'])

## Initialize Transformer-Based Sentiment Analysis Pipeline

We'll use a pre-trained DistilBERT model fine-tuned on the SST-2 sentiment dataset.
This is a state-of-the-art model that provides excellent performance while being relatively lightweight.

In [ ]:
# Initialize the sentiment analysis pipeline with a transformer model
# Using distilbert-base-uncased-finetuned-sst-2-english - a popular and efficient model
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

print(f"Using device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## Alternative: Load Latest RoBERTa Model

Uncomment the following cell to use RoBERTa instead, which is one of the latest and most powerful models:

In [ ]:
# Alternative: Use RoBERTa - one of the latest transformer models
# sentiment_pipeline = pipeline(
#     "sentiment-analysis",
#     model="cardiffnlp/twitter-roberta-base-sentiment-latest",
#     device=0 if torch.cuda.is_available() else -1
# )

## Test the Model on Sample Reviews

In [ ]:
# Test on a few sample reviews
sample_reviews = reviews[:5]
for i, review in enumerate(sample_reviews):
    result = sentiment_pipeline(review[:512])[0]  # Truncate to max length
    print(f"Review {i+1}: {review[:100]}...")
    print(f"Sentiment: {result['label']}, Score: {result['score']:.4f}")
    print("-" * 80)

## Process All Reviews

Convert sentiment labels to numerical scores for comparison with dictionary-based approach:
- POSITIVE: +1
- NEGATIVE: -1
- Score is weighted by the model's confidence

In [ ]:
def get_transformer_sentiment(review, max_length=512):
    """
    Get sentiment score using transformer model.
    Returns a score between -1 and 1.
    """
    # Truncate review to max length to avoid errors
    truncated_review = review[:max_length]
    
    try:
        result = sentiment_pipeline(truncated_review)[0]
        
        # Convert to numerical score: POSITIVE=1, NEGATIVE=-1, weighted by confidence
        if result['label'] == 'POSITIVE':
            return result['score']
        else:  # NEGATIVE
            return -result['score']
    except Exception as e:
        print(f"Error processing review: {e}")
        return 0.0

In [ ]:
# Process all reviews (this may take some time depending on dataset size)
print(f"Processing {len(reviews)} reviews...")
transformer_sentiments = []

for i, review in enumerate(reviews):
    sentiment = get_transformer_sentiment(review)
    transformer_sentiments.append(sentiment)
    
    # Progress indicator
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(reviews)} reviews")

print("Processing complete!")

## Create Results DataFrame

In [ ]:
# Create a DataFrame with results
results_df = pd.DataFrame({
    'rating': ratings,
    'review': reviews,
    'transformer_sentiment': transformer_sentiments
})

results_df.head(10)

## Save Results

In [ ]:
# Save the results
with open('DATA/processed/transformer_based_sentiment.tsv', 'w') as outfile:
    outfile.write(results_df.to_csv(index=False, sep="\t"))

print("Results saved to DATA/processed/transformer_based_sentiment.tsv")

## Analysis and Comparison

In [ ]:
# Basic statistics
print("Transformer Sentiment Statistics:")
print(f"Mean: {np.mean(transformer_sentiments):.4f}")
print(f"Median: {np.median(transformer_sentiments):.4f}")
print(f"Std Dev: {np.std(transformer_sentiments):.4f}")
print(f"Min: {np.min(transformer_sentiments):.4f}")
print(f"Max: {np.max(transformer_sentiments):.4f}")

## Compare with Dictionary-Based Approach (if available)

In [ ]:
# Try to load dictionary-based results for comparison
try:
    dict_df = pd.read_csv('DATA/processed/dictionary_based_sentiment.tsv', sep='\t')
    
    # Merge the results
    comparison_df = results_df.copy()
    comparison_df['dictionary_sentiment'] = dict_df['review dictionary based sentiment']
    
    # Calculate correlation
    correlation = np.corrcoef(
        comparison_df['transformer_sentiment'],
        comparison_df['dictionary_sentiment']
    )[0, 1]
    
    print(f"Correlation between Transformer and Dictionary approaches: {correlation:.4f}")
    
    # Show some examples where they differ significantly
    comparison_df['sentiment_diff'] = abs(
        comparison_df['transformer_sentiment'] - comparison_df['dictionary_sentiment']
    )
    
    print("\nTop 5 reviews with largest difference in sentiment scores:")
    print(comparison_df.nlargest(5, 'sentiment_diff')[[
        'review', 'rating', 'transformer_sentiment', 'dictionary_sentiment', 'sentiment_diff'
    ]])
    
except FileNotFoundError:
    print("Dictionary-based results not found. Run the dictionary-based analyzer first for comparison.")

## Visualization (Optional)

Uncomment and run if you have matplotlib or altair installed:

In [ ]:
# Visualization with altair (if available)
try:
    import altair as alt
    
    # Distribution of transformer sentiments
    hist_data = pd.DataFrame({'sentiment': transformer_sentiments})
    
    chart = alt.Chart(hist_data).mark_bar().encode(
        alt.X('sentiment:Q', bin=alt.Bin(maxbins=50)),
        y='count()'
    ).properties(
        title='Distribution of Transformer-Based Sentiment Scores',
        width=600,
        height=400
    )
    
    chart.save('plots/01/transformer_sentiment_distribution.html')
    print("Visualization saved to plots/01/transformer_sentiment_distribution.html")
    chart
    
except ImportError:
    print("Altair not available. Skipping visualization.")

## Model Information and Performance Notes

**Model Used**: DistilBERT-base-uncased-finetuned-sst-2-english

**Key Advantages of Transformer-Based Approach**:
1. **Contextual Understanding**: Transformers understand word context and relationships
2. **Negation Handling**: Better at understanding negations ("not good" vs "good")
3. **Sarcasm Detection**: Can often detect sarcastic or nuanced sentiments
4. **State-of-the-Art Performance**: Achieves much higher accuracy than dictionary-based methods
5. **Pre-trained Knowledge**: Leverages knowledge from millions of training examples

**Alternative Models to Consider**:
- `roberta-base-sentiment`: More powerful but slower
- `bert-base-uncased`: Original BERT model
- `cardiffnlp/twitter-roberta-base-sentiment-latest`: Fine-tuned on social media text
- `nlptown/bert-base-multilingual-uncased-sentiment`: For multilingual support

To use a different model, simply change the model name in the pipeline initialization.